In [47]:
import pandas as pd
import re

# pandas 3.0 uses new Arrow string type by default which crashes with numpy 2.4
# this setting reverts to old object dtype which is compatible
pd.options.future.infer_string = False

df = pd.read_csv('../data/raw/transactions.csv')
print("Shape:", df.shape)
df.head()

Shape: (4500, 11)


,transaction_id,listing_id,buyer_id,agent_id,sale_price,sale_date,closing_date,commission_amount,transaction_status,financing_type,inspection_passed
0,TXN-00001,LST-2538,BYR-1560,AGT-0014,564343.77,2024-02-19,2024-03-19,18066.11,Cancelled,FHA,False
1,TXN-00002,LST-5244,BYR-0814,AGT-0111,1123508.95,2021-04-15,2021-05-22,65128.04,Cancelled,VA,True
2,TXN-00003,LST-2581,BYR-2677,AGT-0006,2360712.19,2021-04-08,2021-05-11,101742.10,Cancelled,VA,False
3,TXN-00004,LST-5153,BYR-1999,AGT-0108,953489.18,2024-07-31,2024-08-22,42756.33,Cancelled,FHA,True
4,TXN-00005,LST-3511,BYR-1008,AGT-0273,1733871.60,2024-07-20,2024-08-20,99907.22,Pending,Conventional,False


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   transaction_id      4500 non-null   object 
 1   listing_id          4500 non-null   object 
 2   buyer_id            4500 non-null   object 
 3   agent_id            4500 non-null   object 
 4   sale_price          4500 non-null   float64
 5   sale_date           4500 non-null   object 
 6   closing_date        4500 non-null   object 
 7   commission_amount   4500 non-null   float64
 8   transaction_status  4289 non-null   object 
 9   financing_type      4500 non-null   object 
 10  inspection_passed   4500 non-null   bool   
dtypes: bool(1), float64(2), object(8)
memory usage: 356.1+ KB


In [49]:
df.isnull().sum()

transaction_id          0
listing_id              0
buyer_id                0
agent_id                0
sale_price              0
sale_date               0
closing_date            0
commission_amount       0
transaction_status    211
financing_type          0
inspection_passed       0
dtype: int64

In [50]:
df.duplicated().sum()

np.int64(0)

In [51]:
# Step 1: Handle nulls in transaction_status
# 211 rows have no status — filling with 'Unknown' to preserve rows for analysis
print("Null count before:", df['transaction_status'].isnull().sum())

df['transaction_status'] = df['transaction_status'].fillna('Unknown')

print("Null count after :", df['transaction_status'].isnull().sum())
print("\nValue counts after fill:")
print(df['transaction_status'].value_counts())

Null count before: 211
Null count after : 0

Value counts after fill:
transaction_status
Pending      1450
Cancelled    1430
Completed    1409
Unknown       211
Name: count, dtype: int64


In [52]:
# Step 2: Convert sale_date and closing_date from string to datetime
# Required for any date arithmetic, sorting, or time-based validation
print("Dtypes before conversion:")
print(df[['sale_date', 'closing_date']].dtypes)

df['sale_date']     = pd.to_datetime(df['sale_date'],     errors='coerce')
df['closing_date']  = pd.to_datetime(df['closing_date'],  errors='coerce')

print("\nDtypes after conversion:")
print(df[['sale_date', 'closing_date']].dtypes)

# Check if any dates failed to parse (would become NaT)
print(f"\nNaT in sale_date    : {df['sale_date'].isnull().sum()}")
print(f"NaT in closing_date : {df['closing_date'].isnull().sum()}")

print("\nDate ranges:")
print(f"  sale_date    : {df['sale_date'].min().date()} → {df['sale_date'].max().date()}")
print(f"  closing_date : {df['closing_date'].min().date()} → {df['closing_date'].max().date()}")

Dtypes before conversion:
sale_date       object
closing_date    object
dtype: object

Dtypes after conversion:
sale_date       datetime64[ns]
closing_date    datetime64[ns]
dtype: object

NaT in sale_date    : 0
NaT in closing_date : 0

Date ranges:
  sale_date    : 2019-12-08 → 2024-12-26
  closing_date : 2020-01-09 → 2025-01-26


In [53]:
# Step 3: Logical date validation — closing_date must be after sale_date
invalid_dates = df[df['closing_date'] <= df['sale_date']]
print(f"Rows where closing_date <= sale_date: {len(invalid_dates)}")

if len(invalid_dates) > 0:
    print("\nInvalid date rows:")
    print(invalid_dates[['transaction_id', 'sale_date', 'closing_date']])

Rows where closing_date <= sale_date: 0


In [54]:
# Step 4: Validate days-to-close range (expected: 1 to 180 days)
# Deals closing same day or taking over 6 months are likely data errors
df['days_to_close'] = (df['closing_date'] - df['sale_date']).dt.days

print("Days-to-close summary:")
print(df['days_to_close'].describe().round(1))

too_fast = df[df['days_to_close'] < 1]
too_slow = df[df['days_to_close'] > 180]
print(f"\nDeals closed in < 1 day  : {len(too_fast)}")
print(f"Deals closed in > 180 days: {len(too_slow)}")

if len(too_fast) > 0:
    print("\nSample too-fast closings:")
    print(too_fast[['transaction_id', 'sale_date', 'closing_date', 'days_to_close']].head())
if len(too_slow) > 0:
    print("\nSample too-slow closings:")
    print(too_slow[['transaction_id', 'sale_date', 'closing_date', 'days_to_close']].head())

Days-to-close summary:
count    4500.0
mean       29.6
std         8.9
min        15.0
25%        22.0
50%        30.0
75%        37.0
max        45.0
Name: days_to_close, dtype: float64

Deals closed in < 1 day  : 0
Deals closed in > 180 days: 0


In [55]:
# Step 5: Validate sale_price and commission_amount
# Both must be positive; commission rate should fall between 2% and 8% of sale_price
print("sale_price stats:")
print(df['sale_price'].describe().round(2))

print("\ncommission_amount stats:")
print(df['commission_amount'].describe().round(2))

invalid_price      = df[df['sale_price'] <= 0]
invalid_commission = df[df['commission_amount'] <= 0]
print(f"\nRows with sale_price <= 0      : {len(invalid_price)}")
print(f"Rows with commission_amount <= 0: {len(invalid_commission)}")

# Commission rate check (2%–8% is a realistic real estate range)
df['commission_rate'] = (df['commission_amount'] / df['sale_price']) * 100
outside_rate = df[(df['commission_rate'] < 2) | (df['commission_rate'] > 8)]
print(f"Rows with commission rate outside 2%–8%: {len(outside_rate)}")

if len(outside_rate) > 0:
    print("\nSample rows with abnormal commission rates:")
    print(outside_rate[['transaction_id', 'sale_price', 'commission_amount', 'commission_rate']].head())

sale_price stats:
count       4500.00
mean     1377052.81
std       753375.66
min        48236.68
25%       711021.06
50%      1365389.27
75%      1988745.15
max      3195490.35
Name: sale_price, dtype: float64

commission_amount stats:
count      4500.00
mean      58695.34
std       35707.37
min        1546.44
25%       28910.67
50%       55345.60
75%       82264.06
max      181691.44
Name: commission_amount, dtype: float64

Rows with sale_price <= 0      : 0
Rows with commission_amount <= 0: 0
Rows with commission rate outside 2%–8%: 0


In [56]:
# Step 6: Check and standardize transaction_status and financing_type categories
# transaction_status uses title case; financing_type needs a manual map to preserve abbreviations
print("transaction_status unique values:")
print(df['transaction_status'].value_counts())

print("\nfinancing_type unique values:")
print(df['financing_type'].value_counts())

df['transaction_status'] = df['transaction_status'].str.strip().str.title()

# Manual map preserves abbreviations (FHA, VA, USDA) that .title() would break to Fha, Va, Usda
financing_map = {
    'fha': 'FHA', 'va': 'VA', 'usda': 'USDA',
    'conventional': 'Conventional', 'cash': 'Cash', 'jumbo': 'Jumbo'
}
df['financing_type'] = df['financing_type'].str.strip().str.lower().replace(financing_map)

print("\nAfter normalization — transaction_status:")
print(df['transaction_status'].value_counts())
print("\nAfter normalization — financing_type:")
print(df['financing_type'].value_counts())

transaction_status unique values:
transaction_status
Pending      1450
Cancelled    1430
Completed    1409
Unknown       211
Name: count, dtype: int64

financing_type unique values:
financing_type
Conventional    1995
Cash             941
FHA              650
VA               460
Jumbo            350
USDA             104
Name: count, dtype: int64

After normalization — transaction_status:
transaction_status
Pending      1450
Cancelled    1430
Completed    1409
Unknown       211
Name: count, dtype: int64

After normalization — financing_type:
financing_type
Conventional    1995
Cash             941
FHA              650
VA               460
Jumbo            350
USDA             104
Name: count, dtype: int64


In [57]:
# Step 7: ID format validation using regex
# transaction_id: TXN-XXXXX (5 digits), listing_id: LST-XXXX (4 digits)
# buyer_id: BYR-XXXX (4 digits), agent_id: AGT-XXXX (4 digits)
id_patterns = {
    'transaction_id': r'^TXN-\d{5}$',
    'listing_id':     r'^LST-\d{4}$',
    'buyer_id':       r'^BYR-\d{4}$',
    'agent_id':       r'^AGT-\d{4}$',
}

for col, pattern in id_patterns.items():
    invalid = ~df[col].str.match(pattern)
    print(f"Invalid {col}: {invalid.sum()}")
    if invalid.sum() > 0:
        print(df[invalid][col].value_counts().to_string())

# Drop rows with invalid listing_id — placeholder 'LST-XXXX' has no valid listing reference
invalid_listing_mask = ~df['listing_id'].str.match(r'^LST-\d{4}$')
print(f"\nDropping {invalid_listing_mask.sum()} rows with invalid listing_id...")
df = df[~invalid_listing_mask].reset_index(drop=True)
print(f"Shape after drop: {df.shape}")

Invalid transaction_id: 0
Invalid listing_id: 90
listing_id
LST-XXXX    90
Invalid buyer_id: 0
Invalid agent_id: 0

Dropping 90 rows with invalid listing_id...
Shape after drop: (4410, 13)


In [58]:
# Step 8: transaction_id uniqueness check (primary key must have no duplicates)
dupe_txn = df['transaction_id'].duplicated().sum()
print(f"Duplicate transaction_ids: {dupe_txn}")

if dupe_txn > 0:
    print("\nDuplicated transaction_id values:")
    print(df[df['transaction_id'].duplicated(keep=False)].sort_values('transaction_id'))

Duplicate transaction_ids: 0


In [59]:
# Save cleaned data to processed folder
df.to_csv('../data/processed/transactions_cleaned.csv', index=False)
print("Saved: ../data/processed/transactions_cleaned.csv")
print("Final shape:", df.shape)

Saved: ../data/processed/transactions_cleaned.csv
Final shape: (4410, 13)
